## RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\Dell\AppData\Local\Temp\ipykernel_8304\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file)) 
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 5 PDF files to process

Processing: EE3355_L01_4392.pdf
  ✓ Loaded 15 pages

Processing: EE3355_L03_4392.pdf
  ✓ Loaded 3 pages

Processing: EE3355_L1_4385.pdf
  ✓ Loaded 12 pages

Processing: Lab 3.pdf
  ✓ Loaded 12 pages

Processing: Lab_01 Report Format.pdf
  ✓ Loaded 17 pages

Total documents loaded: 59


In [4]:
len(all_pdf_documents)


59

In [5]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=400,chunk_overlap=50):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [6]:
chunks=split_documents(all_pdf_documents)
chunks

Split 59 documents into 96 chunks

Example chunk:
Content: EE3355: SIGNAL AND SYSTEM 
LABORATORY 01 
 
 
 
 
 
 
 
 
 
 
 
 
                                                                        NAME        : AKARSHANA A.N.  
                               ...
Metadata: {'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-11-07T12:35:32+05:30', 'author': 'DEIE', 'moddate': '2026-11-07T12:35:32+05:30', 'source': '..\\data\\pdf\\EE3355_L01_4392.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'EE3355_L01_4392.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-11-07T12:35:32+05:30', 'author': 'DEIE', 'moddate': '2026-11-07T12:35:32+05:30', 'source': '..\\data\\pdf\\EE3355_L01_4392.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'EE3355_L01_4392.pdf', 'file_type': 'pdf'}, page_content='EE3355: SIGNAL AND SYSTEM \nLABORATORY 01 \n \n \n \n \n \n \n \n \n \n \n \n \n                                                                        NAME        : AKARSHANA A.N.  \n                                                                        REG No.     : EE/2021/4392 \n                SEMESTER: 03 \n                                                                        DATE          : 20/11 /2026'),
 Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-11-07T12:35:32+05:30', 'author': 'DEIE', 'moddate': '2026-11-07T12:35:32+05:30', 'source': '..\\data\\pd

## Embedding And VectorStoreDB

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity